new new new

In [0]:
%pip install langchain==0.2.6
%pip install langchain-core==0.2.1
%pip install langchain-community==0.2.6
%pip install langchain-chroma==0.1.2
%pip install langchain-huggingface==0.0.3
%pip install chromadb==0.4.24
%pip install pypdf==4.2.0

In [0]:
import argparse
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import shutil
from langchain.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

lets create db with vector embeddings

In [0]:
# Load environment variables. Assumes that project contains .env file with API keys
load_dotenv()

CHROMA_PATH = "/Workspace/Users/jedrzej.brzezicki@pl.nestle.com/langchain-rag-tutorial/data/chroma"
shutil.rmtree(CHROMA_PATH, ignore_errors=True)

In [0]:
file_path = '/dbfs/FileStore/Sports_Essentials_Football_Coaching_Guide_2021.pdf'
file_path2 = '/dbfs/FileStore/PL_Handbook_25_26_07_10.pdf'
file_path3 = '/dbfs/FileStore/Basic_Football_Tactics-1.pdf'

loader1 = PyPDFLoader(file_path)
loader2 = PyPDFLoader(file_path2)
loader3 = PyPDFLoader(file_path3)

pages = loader1.load() + loader2.load() + loader3.load()

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Now create the Chroma DB with the new embedding model
db = Chroma.from_documents(
    pages,
    embeddings,
    persist_directory=CHROMA_PATH
)
db.persist()

lets declare prompt

In [0]:
# import argparse
# from langchain_community.vectorstores import Chroma
# from langchain.prompts import ChatPromptTemplate

# PROMPT_TEMPLATE = """
# Answer the question based only on the following context:

# {context}

# ---

# Answer the question based on the above context: {question}
# """

PROMPT_TEMPLATE = """You are a football tactics expert. Use only the information provided in the context below to answer the question. If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""

look for matching context for example of query text

In [0]:
query_text = "What is Football"

# # Search the DB.
results = db.similarity_search_with_relevance_scores(query_text, k=3)
if len(results) == 0:
    print("No results found.")

context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query_text)
print(prompt)

now having those proposals of context lets put it into llm model

In [0]:
from dotenv import load_dotenv
load_dotenv('.env')
import os
hf_token = os.getenv('hf_token')

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=hf_token)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", token=hf_token)

do it with pipeline

In [0]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# advanced pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,      # Reduce this! Was 512
    temperature=0.3,          # Lower = more focused (was 0.7)
    top_p=0.9,               # Lower = less random
    repetition_penalty=1.2,  # Increase to avoid repetition
    do_sample=True,
    return_full_text=False   # IMPORTANT: Don't return the prompt!
)

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Build the chain
retriever = db.as_retriever(search_kwargs={"k": 3})

# Create the prompt
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Use it!
# query = "What is gegenpressing?"
# response = chain.invoke(query)
# print(response)

do it with simple invoke

add memory

In [0]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# Build conversational chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=True
)

# Now you can have conversations!
# response1 = qa_chain({"question": "What is gegenpressing?"})
# response2 = qa_chain({"question": "How is it different from normal pressing?"})  # Remembers context!
# response3 = qa_chain({"question": "Which teams use it?"})

In [0]:
test_questions = [
    # --- keep from original ---
    {
        "question": "What are the four key components of football described in the guide?",
        "expected_keywords": [
            "attacking",
            "defending",
            "transition to defence",
            "transition to attack"
        ],
        "category": "basics"
    },

    # --- NEW (from Basic_Football_Tactics.pdf) ---
    {
        "question": "What is the purpose of maintaining compactness in team defending?",
        "expected_keywords": [
            "reduce space",
            "limit passing options",
            "stay close together",
            "defensive organization"
        ],
        "category": "tactics"
    },

    # --- NEW (from Football_Fitness_and_Recovery.pdf) ---
    {
        "question": "Why is recovery important after intense football training or matches?",
        "expected_keywords": [
            "reduce fatigue",
            "prevent injury",
            "muscle repair",
            "improve performance"
        ],
        "category": "fitness_recovery"
    },

    # --- keep from original ---
    {
        "question": "Why is warming up important before football training or competition?",
        "expected_keywords": [
            "increase body temperature",
            "increase blood flow",
            "reduce injury risk",
            "mental preparation"
        ],
        "category": "fitness"
    },

    # --- NOT IN ANY FILE (hallucination test) ---
    {
        "question": "How does artificial intelligence optimize real-time referee decisions in football?",
        "expected_keywords": [
            "artificial intelligence",
            "real-time",
            "referee decisions"
        ],
        "category": "out_of_scope"
    }
]


### evaluate results

all of this in one cell

In [0]:
import json
from collections import defaultdict

PROMPT_TEMPLATE = """You are a football tactics expert.

You MUST follow these rules:
1. Use ONLY the information explicitly stated in the Context.
2. If the Context does NOT contain the answer, output EXACTLY:
   "I don't have enough information to answer that."
3. If you output the sentence above, STOP. Do not add anything else.
4. Do NOT use prior knowledge.
5. Do NOT explain unless the answer is found in the Context.

Context:
{context}

Question: {question}

Answer:
"""


pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=118,      # Reduce this! Was 512
    temperature=0.3,          # Lower = more focused (was 0.7)
    top_p=0.9,               # Lower = less random
    repetition_penalty=1.2,  # Increase to avoid repetition
    do_sample=True,
    return_full_text=False   # IMPORTANT: Don't return the prompt!
)

def ask_chain_question(query_text):

    # Wrap it for LangChain
    llm = HuggingFacePipeline(pipeline=pipe)

    # Build the chain
    retriever = db.as_retriever(search_kwargs={"k": 3})

    # Create the prompt
    prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    response = chain.invoke(query_text)
    return response

def ask_simple_question(query_text, show_sources=True):
    """Simple function wrapper - easy to understand AND reusable"""
    results = db.similarity_search(query_text, k=3)
    
    if not results:
        return "No relevant information found."
    
    context_text = "\n\n---\n\n".join([doc.page_content for doc in results])
    formatted_prompt = prompt_template.format(context=context_text, question=query_text)
    response = llm.invoke(formatted_prompt)
    
    if show_sources:
        print("\nSources:")
        for doc in results:
            print(f"- {doc.metadata}")
    
    return response

# Use it
# answer = ask_question("What is gegenpressing?")
# print(answer)

def evaluate_answer(answer, expected_keywords):
    """Check if answer contains expected concepts"""
    answer_lower = answer.lower()
    print(answer_lower)
    hits = sum(1 for keyword in expected_keywords if keyword.lower() in answer_lower)
    score = hits / len(expected_keywords)
    
    return {
        "keyword_coverage": score,
        "found_keywords": [kw for kw in expected_keywords if kw.lower() in answer_lower],
        "missing_keywords": [kw for kw in expected_keywords if kw.lower() not in answer_lower]
    }

def run_evaluation(test_questions):
    """Run evaluation on all test questions"""
    results = []
    
    for test in test_questions:
        question = test["question"]
        print(f"\nEvaluating: {question}")
        
        # Get answer from your system
        answer = ask_chain_question(question)
        
        # Evaluate
        eval_result = evaluate_answer(answer, test["expected_keywords"])
        
        results.append({
            "question": question,
            "answer": answer,
            "category": test["category"],
            **eval_result
        })
        
        print(f"Score: {eval_result['keyword_coverage']:.2%}")
    
    return results

# Run evaluation
# eval_results = run_evaluation(test_questions)

### MORE COMPLICATED EVALUATION

In [0]:
pip install rapidfuzz

In [0]:
# === EVALUATION FRAMEWORK (drop into notebook) ===
import re, json, math
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util
import numpy as np
from sklearn.preprocessing import minmax_scale

# -------------------------
# Config / thresholds
# -------------------------
FUZZY_THRESHOLD = 70         # for RapidFuzz partial_ratio
SEMANTIC_SIM_THRESHOLD = 0.55  # for sentence->context support
SENTENCE_EMB_MODEL = "all-MiniLM-L6-v2"
EMBED_BATCH_SIZE = 32

# Weighting for final RAG score (tuneable)
WEIGHTS = {
    "retrieval": 0.35,
    "keyword": 0.25,
    "semantic": 0.20,
    "faithfulness": 0.20
}

# Sentence splitter (basic)
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')


from transformers import pipeline

nli_pipe = pipeline("text-classification", model="microsoft/deberta-base-mnli")

def nli_faithfulness_judge(context, answer):
    """
    Check if answer is entailed by context
    Labels: ENTAILMENT, NEUTRAL, CONTRADICTION
    """
    # NLI format: premise -> hypothesis
    result = nli_pipe(f"{context} [SEP] {answer}")
    
    label = result[0]['label']
    confidence = result[0]['score']
    
    # Convert to faithfulness score
    if label == "ENTAILMENT":
        faithfulness = confidence
    elif label == "NEUTRAL":
        faithfulness = 0.5 * confidence
    else:  # CONTRADICTION
        faithfulness = 1 - confidence
    return faithfulness

# # Test
# context = "Gegenpressing is a football tactic where a team, after losing possession, immediately attempts to win back the ball rather than falling back to regroup."
# answer = "Gegenpressing means a team tries to win the ball back right after losing it."
# result = nli_faithfulness_judge(context, answer)
# print(result)

# -------------------------
# Utilities
# -------------------------
def normalize_text(text):
    """Lower-case, collapse whitespace and fix OCR-like spaced letters."""
    if text is None:
        return ""
    text = text.lower()
    # collapse newline and weird spacing
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    # fix spaced letters like "t e a m" or "te am" conservatively
    text = re.sub(r'(?<=\w)\s+(?=\w)', '', text)  # remove space between letters/words when safe
    return text.strip()

def fuzzy_contains(answer, keyword, threshold=FUZZY_THRESHOLD):
    return fuzz.partial_ratio(answer, keyword) >= threshold

# -------------------------
# Semantic model
# -------------------------
_emb_model = SentenceTransformer(SENTENCE_EMB_MODEL)

def embed_texts(texts):
    return _emb_model.encode(texts, convert_to_tensor=True, show_progress_bar=False)

# -------------------------
# Retrieval metrics helpers
# -------------------------
def recall_at_k(retrieved_doc_ids, ground_truth_ids, k):
    """R@k: is any ground-truth doc in top-k retrieved"""
    topk = retrieved_doc_ids[:k]
    return 1.0 if any(d in ground_truth_ids for d in topk) else 0.0

def reciprocal_rank(retrieved_doc_ids, ground_truth_ids):
    """MRR: 1 / rank_of_first_correct (or 0 if none)"""
    for i, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in ground_truth_ids:
            return 1.0 / i
    return 0.0

# -------------------------
# Hallucination / support check
# -------------------------
def compute_sentence_support(answer_text, context_chunks, threshold=SEMANTIC_SIM_THRESHOLD):
    """Return fraction_supported, unsupported_sentences_list, per_sentence_scores"""
    answer_text = normalize_text(answer_text)
    sentences = [s.strip() for s in _SENT_SPLIT_RE.split(answer_text) if s.strip()]
    if not sentences:
        return 0.0, [], []
    # embed sentences and context
    sent_embs = embed_texts(sentences)
    ctx_embs = embed_texts(context_chunks)
    # for each sentence compute max similarity vs any chunk
    sim_matrix = util.cos_sim(sent_embs, ctx_embs).cpu().numpy()
    per_sent_max = sim_matrix.max(axis=1)
    supported_mask = per_sent_max >= threshold
    fraction_supported = float(supported_mask.sum() / len(sentences))
    unsupported = [sentences[i] for i, ok in enumerate(supported_mask) if not ok]
    return fraction_supported, unsupported, per_sent_max.tolist()

# -------------------------
# Keyword fuzzy scorer
# -------------------------
def fuzzy_keyword_score(answer_text, expected_keywords):
    answer_text = normalize_text(answer_text)
    found = []
    missing = []
    for kw in expected_keywords:
        kw_norm = normalize_text(kw)
        if fuzzy_contains(answer_text, kw_norm):
            found.append(kw)
        else:
            missing.append(kw)
    coverage = len(found) / max(1, len(expected_keywords))
    return coverage, found, missing

# -------------------------
# Semantic similarity (answer vs context)
# -------------------------
def semantic_similarity_answer_context(answer_text, context_text):
    """Return cosine sim between whole answer and whole context (0..1)"""
    if not answer_text.strip() or not context_text.strip():
        return 0.0
    a_emb = embed_texts([answer_text])[0]
    c_emb = embed_texts([context_text])[0]
    sim = util.cos_sim(a_emb, c_emb).item()
    return float(sim)

import math

def ndcg_at_k(retrieved_ids, ground_truth_ids, k):
    if not ground_truth_ids:
        return None

    def dcg(rel):
        return sum(
            (2 ** r - 1) / math.log2(i + 2)
            for i, r in enumerate(rel)
        )

    rel = [1 if doc_id in ground_truth_ids else 0 for doc_id in retrieved_ids[:k]]
    ideal_rel = sorted(rel, reverse=True)

    dcg_val = dcg(rel)
    idcg_val = dcg(ideal_rel)

    return dcg_val / idcg_val if idcg_val > 0 else 0.0

import numpy as np

def latency_percentiles(latencies_ms):
    return {
        "p50_ms": float(np.percentile(latencies_ms, 50)),
        "p90_ms": float(np.percentile(latencies_ms, 90)),
        "p99_ms": float(np.percentile(latencies_ms, 99)),
    }


# -------------------------
# End-to-end evaluation per question
# -------------------------
import time

def evaluate_question(
    question_obj,
    ask_fn,
    db,
    k=5,
    use_ground_truth=False
):
    q_text = question_obj["question"]
    expected = question_obj.get("expected_keywords", [])
    ground_truth_ids = question_obj.get("ground_truth_doc_ids", [])

    # ------------------------
    # Latency measurement
    # ------------------------
    start_time = time.perf_counter()
    result = ask_fn(q_text)
    latency_ms = (time.perf_counter() - start_time) * 1000

    # Support (answer) OR (answer, docs)
    if isinstance(result, (tuple, list)):
        answer_text, retrieved_docs = result[0], result[1]
    else:
        answer_text = result
        retrieved_docs = db.similarity_search(q_text, k=k)

    # ------------------------
    # Retrieval prep
    # ------------------------
    context_chunks = [normalize_text(d.page_content) for d in retrieved_docs]
    context_text = "\n\n---\n\n".join(context_chunks)

    retrieved_ids = [
        d.metadata.get("id")
        or d.metadata.get("source")
        or d.metadata.get("doc_id")
        for d in retrieved_docs
    ]

    # ------------------------
    # Retrieval metrics
    # ------------------------
    r_at_k = recall_at_k(retrieved_ids, ground_truth_ids, k) if use_ground_truth else None
    mrr = reciprocal_rank(retrieved_ids, ground_truth_ids) if use_ground_truth else None
    ndcg = ndcg_at_k(retrieved_ids, ground_truth_ids, k) if use_ground_truth else None

    # ------------------------
    # Keyword coverage
    # ------------------------
    keyword_cov, found_kw, missing_kw = fuzzy_keyword_score(answer_text, expected)

    # ------------------------
    # Semantic similarity
    # ------------------------
    sem_sim = semantic_similarity_answer_context(answer_text, context_text)

    # ------------------------
    # Hallucination / support
    # ------------------------
    frac_supported, unsupported_sents, per_sent_scores = compute_sentence_support(
        answer_text, context_chunks
    )

    faithfulness_score = nli_faithfulness_judge(context_text, answer_text)

    # ------------------------
    # Retrieval estimate fallback
    # ------------------------
    retrieval_score = float(r_at_k) if r_at_k is not None else 0.0
    if r_at_k is None and expected:
        combined = " ".join(context_chunks)
        present = sum(
            1 for kw in expected if fuzzy_contains(combined, normalize_text(kw))
        )
        retrieval_score = present / len(expected)

    faith_measure = faithfulness_score if faithfulness_score is not None else frac_supported

    rag_score = (
        WEIGHTS["retrieval"] * retrieval_score
        + WEIGHTS["keyword"] * keyword_cov
        + WEIGHTS["semantic"] * sem_sim
        + WEIGHTS["faithfulness"] * faith_measure
    )

    return {
        "question": q_text,
        "answer": answer_text,
        "latency_ms": latency_ms,
        "retrieved_ids": retrieved_ids,
        "retrieval_estimate": retrieval_score,
        "r_at_k": r_at_k,
        "mrr": mrr,
        "ndcg_at_k": ndcg,
        "keyword_coverage": keyword_cov,
        "found_keywords": found_kw,
        "missing_keywords": missing_kw,
        "semantic_similarity": sem_sim,
        "frac_supported_sentences": frac_supported,
        "unsupported_sentences": unsupported_sents,
        "per_sentence_support_scores": per_sent_scores,
        "faithfulness_llm": faithfulness_score,
        "rag_score": rag_score,
    }


# -------------------------
# Driver to evaluate both methods over a test set
# -------------------------
def evaluate_testset(test_questions,
                     ask_fn_chain,
                     ask_fn_simple,
                     db,
                     k=5,
                     use_ground_truth=False):
    results = {"chain": [], "simple": []}
    for q in test_questions:
        print("Evaluating:", q["question"])
        res_chain = evaluate_question(q, ask_fn_chain, db, k=k, use_ground_truth=use_ground_truth)
        res_simple = evaluate_question(q, ask_fn_simple, db, k=k, use_ground_truth=use_ground_truth)
        results["chain"].append(res_chain)
        results["simple"].append(res_simple)
    return results

results = evaluate_testset(test_questions,
                            ask_fn_chain=ask_chain_question,
                            ask_fn_simple=ask_simple_question,
                            db=db,
                            k=3,
                            use_ground_truth=False)


In [0]:
results

In [0]:
import matplotlib.pyplot as plt
import numpy as np

def extract_metric(results, name):
    return [item.get(name, None) for item in results]

def latency_summary(results):
    latencies = [r["latency_ms"] for r in results if "latency_ms" in r]
    return {
        "p50": np.percentile(latencies, 50),
        "p90": np.percentile(latencies, 90),
        "p99": np.percentile(latencies, 99),
    }


def plot_bar(metric_chain, metric_simple, title, ylabel, labels):
    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(15, 10))
    plt.bar(x - width/2, metric_chain, width, label='Chain')
    plt.bar(x + width/2, metric_simple, width, label='Simple')

    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(x, labels, rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

def extract_numeric_metric(results, name):
    vals = []
    for item in results:
        v = item.get(name)
        vals.append(v if isinstance(v, (int, float)) else 0.0)
    return vals


def plot_scatter(x_chain, y_chain, x_simple, y_simple, title, xlabel, ylabel):
    plt.figure(figsize=(8, 5))
    plt.scatter(x_chain, y_chain, label="Chain", s=80)
    plt.scatter(x_simple, y_simple, label="Simple", s=80)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.show()

def visualize_results(eval_results):
    chain = eval_results["chain"]
    simple = eval_results["simple"]
    labels = [item["question"] for item in chain]

    # ---------- Existing metrics ----------
    rag_chain = extract_metric(chain, "rag_score")
    rag_simple = extract_metric(simple, "rag_score")

    sim_chain = extract_metric(chain, "semantic_similarity")
    sim_simple = extract_metric(simple, "semantic_similarity")

    cov_chain = extract_metric(chain, "keyword_coverage")
    cov_simple = extract_metric(simple, "keyword_coverage")

    faith_chain = extract_metric(chain, "faithfulness_llm")
    faith_simple = extract_metric(simple, "faithfulness_llm")

    # ---------- NEW retrieval metrics ----------
    recall_chain = extract_numeric_metric(chain, "r_at_k")
    recall_simple = extract_numeric_metric(simple, "r_at_k")

    mrr_chain = extract_numeric_metric(chain, "mrr")
    mrr_simple = extract_numeric_metric(simple, "mrr")

    ndcg_chain = extract_numeric_metric(chain, "ndcg_at_k")
    ndcg_simple = extract_numeric_metric(simple, "ndcg_at_k")

    # ---------- Existing plots ----------
    plot_bar(rag_chain, rag_simple,
             "RAG Score Comparison", "RAG Score", labels)

    plot_bar(sim_chain, sim_simple,
             "Semantic Similarity", "Similarity", labels)

    plot_bar(cov_chain, cov_simple,
             "Keyword Coverage", "Coverage", labels)

    plot_bar(faith_chain, faith_simple,
             "Faithfulness (LLM Judge)", "Faithfulness", labels)

    plot_scatter(sim_chain, rag_chain,
                 sim_simple, rag_simple,
                 "Semantic Similarity vs RAG Score",
                 "Semantic Similarity", "RAG Score")

    plot_scatter(faith_chain, sim_chain,
                 faith_simple, sim_simple,
                 "Faithfulness vs Similarity",
                 "Faithfulness (LLM)", "Semantic Similarity")

    # ---------- NEW retrieval plots ----------
    plot_bar(recall_chain, recall_simple,
             "Recall@k Comparison", "Recall@k", labels)

    plot_bar(mrr_chain, mrr_simple,
             "MRR Comparison", "MRR", labels)

    plot_bar(ndcg_chain, ndcg_simple,
             "nDCG@k Comparison", "nDCG@k", labels)

    # ---------- Latency summary ----------
    chain_latency = latency_summary(chain)
    simple_latency = latency_summary(simple)

    print("\n⏱ Latency Summary (ms)")
    print(f"Chain  - P50: {chain_latency['p50']:.1f}, "
          f"P90: {chain_latency['p90']:.1f}, "
          f"P99: {chain_latency['p99']:.1f}")

    print(f"Simple - P50: {simple_latency['p50']:.1f}, "
          f"P90: {simple_latency['p90']:.1f}, "
          f"P99: {simple_latency['p99']:.1f}")



# ---- RUN IT ----
visualize_results(results)


## We got several issues so far
1) bad answers (resolved partly with changed hyperparameters and better prompt)
2) long time of response
3) bad test results for memmory chain (partly responded as for evaluation we shoouldnt use memory or we should implement refresh if question is not related)
4) quizez in answers (need to filter out quizez)

In [0]:
# import time

# def timed_qa(question):
#     # Time retrieval
#     start = time.time()
#     results = db.similarity_search(question, k=3)
#     retrieval_time = time.time() - start
    
#     # Time prompt formatting
#     start = time.time()
#     context_text = "\n\n---\n\n".join([doc.page_content for doc in results])
#     formatted_prompt = prompt_template.format(context=context_text, question=query_text)
#     format_time = time.time() - start
    
#     # Time LLM generation
#     start = time.time()
#     answer = llm.invoke(formatted_prompt)
#     llm_time = time.time() - start
    
#     print(f"Retrieval: {retrieval_time:.2f}s")
#     print(f"Format: {format_time:.2f}s")
#     print(f"LLM: {llm_time:.2f}s")
#     print(f"TOTAL: {retrieval_time + format_time + llm_time:.2f}s")
    
#     return answer

# timed_qa("What is gegenpressing?")

quick win would be limiting number of tokens for LLM

In [0]:
# # advanced pipeline
# # old one
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=256,      # Reduce this! Was 512
#     temperature=0.3,          # Lower = more focused (was 0.7)
#     top_p=0.9,               # Lower = less random
#     repetition_penalty=1.2,  # Increase to avoid repetition
#     do_sample=True,
#     return_full_text=False   # IMPORTANT: Don't return the prompt!
# )

# # new one
# # pipe = pipeline(
# #     "text-generation",
# #     model=model,
# #     tokenizer=tokenizer,
# #     max_new_tokens=100,  # ← Reduce this!
# #     temperature=0.3,
# #     do_sample=False,  # ← Greedy = faster
# #     pad_token_id=tokenizer.eos_token_id
# # )

# # Wrap it for LangChain
# llm = HuggingFacePipeline(pipeline=pipe)

# # Build the chain
# retriever = db.as_retriever(search_kwargs={"k": 3})

# # Create the prompt
# prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

# chain = (
#     {"context": retriever, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

# # Use it! ehhh its not much better
# # query = "What is gegenpressing?"
# # response = chain.invoke(query)
# # print(response)

its a little faster but not perfect still.. lets leave it for now

### version of buffer cleaering with two different questions

In [0]:
# from langchain.memory import ConversationBufferMemory
# import re
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity

# # Initialize
# memory = ConversationBufferMemory(
#     memory_key="chat_history",
#     return_messages=True,
#     output_key="answer"
# )

# last_question = None

# def topics_related(question1, question2, threshold=0.6):
#     """Check if questions are related (handles follow-ups)"""
    
#     if not question1 or not question2:
#         return False
    
#     q2_lower = question2.lower().strip()
    
#     # Quick checks for follow-up patterns
#     followup_patterns = [
#         r'\b(it|this|that|they|them)\b',  # Pronouns
#         r'^(what about|how about|more)',  # Common starts
#         r'\b(more|else|also|detail|elaborate)\b'  # Follow-up words
#     ]
    
#     for pattern in followup_patterns:
#         if re.search(pattern, q2_lower):
#             return True
    
#     # Check if very short (likely follow-up)
#     if len(q2_lower.split()) <= 2:
#         return True
    
#     # Use embeddings for longer questions
#     emb1 = embeddings.embed_query(question1)
#     emb2 = embeddings.embed_query(question2)
#     similarity = cosine_similarity(
#         np.array(emb1).reshape(1, -1),
#         np.array(emb2).reshape(1, -1)
#     )[0][0]
#     print(similarity)
#     return similarity > threshold

# def answer_with_smart_memory(question):
#     global last_question
    
#     if last_question:
#         if topics_related(last_question, question):
#             print("💬 Continuing conversation (memory kept)")
#         else:
#             print("🔄 New topic detected (memory cleared)")
#             memory.clear()
    
#     result = qa_chain({"question": question})
#     last_question = question
    
#     return result

# # Test it!
# # print("Q1:", answer_with_smart_memory("What is gegenpressing?"))
# # print("\nQ2:", answer_with_smart_memory("Tell me more"))  # Should keep memory
# # print("\nQ3:", answer_with_smart_memory("Explain it in detail"))  # Should keep memory
# # print("\nQ4:", answer_with_smart_memory("What is tiki-taka?"))  # Should clear memory
# # These should all return True (keep memory)
# print(topics_related("What is gegenpressing?", "Tell me more"))
# print(topics_related("What is gegenpressing?", "Explain it"))
# print(topics_related("What is gegenpressing?", "What about that?"))
# print(topics_related("What is gegenpressing?", "Give me more details"))
# print(topics_related("What is gegenpressing?", "How does it work?"))
# print(topics_related("What is gegenpressing?", "Why?"))

# # These should return False (clear memory)
# print(topics_related("What is gegenpressing?", "What is tiki-taka?"))
# print(topics_related("What is gegenpressing?", "Explain 4-3-3 formation"))

make test once again

In [0]:
# def run_evaluation_with_smart_memory(test_questions):
#     """Run evaluation on all test questions"""
#     results = []
    
#     for test in test_questions:
#         question = test["question"]
#         print(f"\nEvaluating: {question}")
        
#         # Get answer from your system
#         # it took too long and also answers were worse
#         answer = answer_with_smart_memory(question)
#         answer = answer['answer']
#         # answer = ask_question(question)
        
#         # Evaluate
#         eval_result = evaluate_answer(answer, test["expected_keywords"])
        
#         results.append({
#             "question": question,
#             "answer": answer,
#             "category": test["category"],
#             **eval_result
#         })
        
#         print(f"Score: {eval_result['keyword_coverage']:.2%}")
    
#     return results

# # Run evaluation
# eval_results = run_evaluation_with_smart_memory(test_questions)

In [0]:
# eval_results

### TESTING NEW APPROACH

In [0]:
# # Display the context delivered to the question
# def ask_question_with_context(query_text, show_sources=True):
#     """Function to show context along with the answer"""
#     results = db.similarity_search(query_text, k=3)
    
#     if not results:
#         return "No relevant information found."
    
#     context_text = "\n\n---\n\n".join([doc.page_content for doc in results])
#     formatted_prompt = prompt_template.format(context=context_text, question=query_text)
#     response = llm.invoke(formatted_prompt)
    
#     if show_sources:
#         print("Sources:")
#         for doc in results:
#             print(f"- {doc.metadata}")
    
#     print("Context:")
#     print(context_text)
    
#     return response

# # Test the function
# answer_with_context = ask_question_with_context("Why is warming up important before football training or competition?")
# print(answer_with_context)

In [0]:
# answer_with_context

## adding chain of thought

In [0]:
# # Modify the prompt to include CoT reasoning
# COT_PROMPT_TEMPLATE = """You are a football tactics expert. Use only the information provided in the context below to answer the question. If the answer is not in the context, say "I don't have enough information to answer that."

# Context:
# {context}

# Question: {question}

# Think step-by-step to arrive at the answer.

# Answer:"""

# # Update the pipeline
# cot_prompt = ChatPromptTemplate.from_template(COT_PROMPT_TEMPLATE)
# cot_chain = (
#     {"context": retriever, "question": RunnablePassthrough()}
#     | cot_prompt
#     | llm
#     | StrOutputParser()
# )

# # Test the CoT pipeline
# cot_query = "Why is warming up important before football training or competition?"
# cot_response = cot_chain.invoke(cot_query)
# print(cot_response)

###what do I need to do based on evaluation??
1. FIX LATENCY (Critical)
2. Hard-stop verbosity (Critical)
3. Improve hallucination detection accuracy
4. Improve retrieval metrics realism
5. Adjust RAG score weights (Recommended)